In [ ]:
"""
Notebook untuk mengambil data berita dari Elasticsearch dan mempersiapkan dataset NER.
"""

In [ ]:
from elasticsearch import Elasticsearch
import pandas as pd
from dotenv import load_dotenv
import os

load_dotenv()


In [ ]:
ES_HOST = os.getenv("ELASTICSEARCH_HOST")
ES_USER = os.getenv("ELASTICSEARCH_USER")
ES_PASSWORD = os.getenv("ELASTICSEARCH_PASSWORD")

es = Elasticsearch(
    ES_HOST,
    basic_auth=(ES_USER, ES_PASSWORD),
    request_timeout=60
)

res = es.search(
    index="news_2025.07",
    query={"match_all": {}},
    size=400
)

docs = [hit["_source"] for hit in res["hits"]["hits"]]
df = pd.DataFrame(docs)


In [ ]:
news_summary = df[['hashtitle','summary', 'ners']]

def extract_entity(ners_dict, entity_type):
    try:
        if isinstance(ners_dict, dict):
            return ners_dict.get(entity_type, [])
        elif isinstance(ners_dict, str):
            ners_dict = ast.literal_eval(ners_dict)
            return ners_dict.get(entity_type, [])
        else:
            return []
    except:
        return []

news_summary['location'] = news_summary['ners'].apply(lambda x: extract_entity(x, 'location'))
news_summary['organization'] = news_summary['ners'].apply(lambda x: extract_entity(x, 'organization'))
news_summary['person'] = news_summary['ners'].apply(lambda x: extract_entity(x, 'person'))

news_summary.drop('ners', axis=1)

# Rename kolom
news_summary = news_summary.rename(columns={
    'location': 'old_LOC',
    'organization': 'old_ORG',
    'person': 'old_PER'
})

news_summary['gt_PER'] = [[] for _ in range(len(news_summary))]
news_summary['gt_LOC'] = [[] for _ in range(len(news_summary))]
news_summary['gt_ORG'] = [[] for _ in range(len(news_summary))]

news_summary = news_summary.drop(columns=['old_LOC', 'old_ORG', 'old_PER'], errors='ignore')


In [ ]:
news_summary.to_csv('news_2025.csv', index=False)